# 🧭 Bússola Pública — Etapa 1: Extração da API da Câmara

**Responsável:** Integrante 2 — Engenharia de Dados

Este notebook cobre **apenas a extração** dos retornos da [API de Dados Abertos da Câmara](https://dadosabertos.camara.leg.br/swagger/api.html):

| Endpoint | O que traz |
| :--- | :--- |
| `/partidos` | Partidos em exercício |
| `/deputados` | Deputados da legislatura atual |
| `/proposicoes` | Proposições apresentadas na janela de datas |
| `/votacoes` | Votações realizadas na janela de datas |
| `/deputados/{id}/despesas` | Despesas da cota parlamentar (por deputado) |

### Como a paginação funciona
A API devolve no máximo **100 itens por página**. Cada resposta traz um bloco `links` com `self`, `first`, `last` e — enquanto houver mais dados — **`next`**.
Por isso a extração roda em um **`while`**: busca a página, acumula os `dados` e segue o link `next` até ele não existir mais.

## 1. Imports e configuração

In [1]:
import time
from datetime import date, timedelta

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 80)

BASE_URL = "https://dadosabertos.camara.leg.br/api/v2"
ITENS_POR_PAGINA = 100      # máximo aceito pela API
PAUSA_ENTRE_PAGINAS = 0.2   # segundos — gentileza com a API (rate limit)

# Janela de extração: últimos 30 dias
DATA_FIM = date.today()
DATA_INICIO = DATA_FIM - timedelta(days=30)

print(f"Janela de extração: {DATA_INICIO} → {DATA_FIM}")

Janela de extração: 2026-10-27 → 2026-11-26


## 2. Sessão HTTP com *retry*
Uma única `Session` reaproveita a conexão. O `Retry` repete automaticamente em erros transitórios (`429` rate limit, `5xx`) com *backoff* exponencial.

In [2]:
def criar_sessao() -> requests.Session:
    retry = Retry(
        total=5,
        backoff_factor=1,                       # 1s, 2s, 4s, 8s...
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        respect_retry_after_header=True,
    )
    sessao = requests.Session()
    sessao.mount("https://", HTTPAdapter(max_retries=retry))
    sessao.headers.update({"Accept": "application/json", "User-Agent": "bussola-publica-etl/1.0"})
    return sessao

sessao = criar_sessao()

## 3. Função de extração paginada (`while`)
Fluxo:
1. Monta a URL da primeira página com os parâmetros.
2. **Enquanto** houver URL: faz o GET, acumula `dados`, procura o link `next`.
3. Quando não há `next`, o loop termina.

O link `next` já vem com todos os parâmetros (incluindo `pagina=N`), então nas páginas seguintes não reenviamos `params`.

In [3]:
def proximo_link(links: list) -> str | None:
    """Retorna o href do link 'next', ou None se for a última página."""
    for link in links:
        if link.get("rel") == "next":
            return link.get("href")
    return None


def extrair_paginado(endpoint: str, params: dict | None = None, verbose: bool = True) -> list[dict]:
    """Percorre todas as páginas de um endpoint da API da Câmara e devolve a lista de registros."""
    url = f"{BASE_URL}/{endpoint}"
    params = {**(params or {}), "itens": ITENS_POR_PAGINA}
    registros: list[dict] = []
    pagina = 0
    inicio = time.time()

    while url:
        resposta = sessao.get(url, params=params, timeout=60)
        resposta.raise_for_status()
        corpo = resposta.json()

        dados = corpo.get("dados", [])
        registros.extend(dados)
        pagina += 1

        if verbose:
            print(f"  [{endpoint}] página {pagina:>3} → {len(dados):>3} itens (acumulado: {len(registros)})")

        # Segue o 'next'; se a página veio vazia, para (proteção contra loop infinito)
        url = proximo_link(corpo.get("links", [])) if dados else None
        params = None  # o link 'next' já carrega os parâmetros
        if url:
            time.sleep(PAUSA_ENTRE_PAGINAS)

    print(f"✅ {endpoint}: {len(registros)} registros em {pagina} página(s) — {time.time() - inicio:.1f}s")
    return registros

## 4. Partidos — `/partidos`

In [4]:
raw_partidos = extrair_paginado("partidos")
df_partidos = pd.DataFrame(raw_partidos)
print(df_partidos.shape)
df_partidos.head()

  [partidos] página   1 →  21 itens (acumulado: 21)
✅ partidos: 21 registros em 1 página(s) — 0.4s
(21, 4)


,id,sigla,nome,uri
0,36898,AVANTE,Avante,https://dadosabertos.camara.leg.br/api/v2/partidos/36898
1,37905,CIDADANIA,Cidadania,https://dadosabertos.camara.leg.br/api/v2/partidos/37905
2,36899,MDB,Movimento Democrático Brasileiro,https://dadosabertos.camara.leg.br/api/v2/partidos/36899
3,38011,MISSÃO,Partido Missão,https://dadosabertos.camara.leg.br/api/v2/partidos/38011
4,37901,NOVO,Partido Novo,https://dadosabertos.camara.leg.br/api/v2/partidos/37901


## 5. Deputados — `/deputados`

In [5]:
raw_deputados = extrair_paginado("deputados")
df_deputados = pd.DataFrame(raw_deputados)
print(df_deputados.shape)
df_deputados.head()

  [deputados] página   1 → 100 itens (acumulado: 100)


  [deputados] página   2 → 100 itens (acumulado: 200)


  [deputados] página   3 → 100 itens (acumulado: 300)


  [deputados] página   4 → 100 itens (acumulado: 400)


  [deputados] página   5 → 100 itens (acumulado: 500)


  [deputados] página   6 →  13 itens (acumulado: 513)
✅ deputados: 513 registros em 6 página(s) — 3.5s
(513, 9)


,id,uri,nome,siglaPartido,uriPartido,siglaUf,idLegislatura,urlFoto,email
0,204379,https://dadosabertos.camara.leg.br/api/v2/deputados/204379,Acácio Favacho,MDB,https://dadosabertos.camara.leg.br/api/v2/partidos/36899,AP,57,https://www.camara.leg.br/internet/deputado/bandep/204379.jpg,dep.acaciofavacho@camara.leg.br
1,220714,https://dadosabertos.camara.leg.br/api/v2/deputados/220714,Adail Filho,MDB,https://dadosabertos.camara.leg.br/api/v2/partidos/36899,AM,57,https://www.camara.leg.br/internet/deputado/bandep/220714.jpg,dep.adailfilho@camara.leg.br
2,221328,https://dadosabertos.camara.leg.br/api/v2/deputados/221328,Adilson Barroso,PL,https://dadosabertos.camara.leg.br/api/v2/partidos/37906,SP,57,https://www.camara.leg.br/internet/deputado/bandep/221328.jpg,dep.adilsonbarroso@camara.leg.br
3,204560,https://dadosabertos.camara.leg.br/api/v2/deputados/204560,Adolfo Viana,PSDB,https://dadosabertos.camara.leg.br/api/v2/partidos/36835,BA,57,https://www.camara.leg.br/internet/deputado/bandep/204560.jpg,dep.adolfoviana@camara.leg.br
4,204528,https://dadosabertos.camara.leg.br/api/v2/deputados/204528,Adriana Ventura,NOVO,https://dadosabertos.camara.leg.br/api/v2/partidos/37901,SP,57,https://www.camara.leg.br/internet/deputado/bandep/204528.jpg,dep.adrianaventura@camara.leg.br


## 6. Proposições — `/proposicoes`
Este é o endpoint com **mais páginas** (dezenas de páginas para 30 dias). `ordenarPor=id` deixa a ordem estável entre as páginas.

In [6]:
raw_proposicoes = extrair_paginado(
    "proposicoes",
    params={
        "dataApresentacaoInicio": DATA_INICIO.isoformat(),
        "dataApresentacaoFim": DATA_FIM.isoformat(),
        "ordem": "ASC",
        "ordenarPor": "id",
    },
    verbose=True,
)
df_proposicoes = pd.DataFrame(raw_proposicoes)
print(df_proposicoes.shape)
df_proposicoes.head()

  [proposicoes] página   1 → 100 itens (acumulado: 100)

  [proposicoes] página   2 → 100 itens (acumulado: 200)


  [proposicoes] página   3 → 100 itens (acumulado: 300)


  [proposicoes] página   4 → 100 itens (acumulado: 400)


  [proposicoes] página   5 → 100 itens (acumulado: 500)


  [proposicoes] página   6 → 100 itens (acumulado: 600)


... (67 outputs left out)

  [proposicoes] página  74 → 100 itens (acumulado: 7400)


  [proposicoes] página  75 → 100 itens (acumulado: 7500)


  [proposicoes] página  76 →  14 itens (acumulado: 7514)
✅ proposicoes: 7514 registros em 76 página(s) — 33.1s
(7514, 8)


,id,uri,siglaTipo,codTipo,numero,ano,ementa,dataApresentacao
0,2648935,https://dadosabertos.camara.leg.br/api/v2/proposicoes/2648935,PRL,190,1,0,"Parecer do Relator, Dep. Diego Garcia (UNIÃO-PR), pela aprovação, com emenda.",2026-10-29T09:07
1,2648936,https://dadosabertos.camara.leg.br/api/v2/proposicoes/2648936,EMR,257,1,0,Reconhece como manifestação da cultura nacional o Festejo do Santuário Dioce...,2026-10-29T09:07
2,2648938,https://dadosabertos.camara.leg.br/api/v2/proposicoes/2648938,PRL,190,1,0,"Parecer do Relator, Dep. Cleber Verde (MDB-MA), pela incompatibilidade e ina...",2026-10-27T10:13
3,2648939,https://dadosabertos.camara.leg.br/api/v2/proposicoes/2648939,PL,139,5663,2026,Dispõe sobre medidas de proteção à mulher em situação de violência doméstica...,2026-10-27T10:23
4,2648940,https://dadosabertos.camara.leg.br/api/v2/proposicoes/2648940,PL,139,5664,2026,"Altera a Lei nº 9.797, de 6 de maio de 1999, para instituir o Kit de Reabili...",2026-10-27T10:23


## 7. Votações — `/votacoes`
⚠️ **Pegadinha da API:** em `/votacoes` o `dataFim` **não inclui o dia inteiro** (testado: em 10/11 vieram só 4 de 294 votações).
Por isso enviamos `dataFim + 1 dia` e depois filtramos pela coluna `data` para manter exatamente a janela.

In [7]:
raw_votacoes = extrair_paginado(
    "votacoes",
    params={
        "dataInicio": DATA_INICIO.isoformat(),
        "dataFim": (DATA_FIM + timedelta(days=1)).isoformat(),  # dataFim é exclusivo na prática
    },
)
raw_votacoes = [v for v in raw_votacoes if DATA_INICIO.isoformat() <= v["data"] <= DATA_FIM.isoformat()]
df_votacoes = pd.DataFrame(raw_votacoes)
print(df_votacoes.shape)
df_votacoes.head()

  [votacoes] página   1 → 100 itens (acumulado: 100)


  [votacoes] página   2 → 100 itens (acumulado: 200)


  [votacoes] página   3 → 100 itens (acumulado: 300)


  [votacoes] página   4 → 100 itens (acumulado: 400)


  [votacoes] página   5 → 100 itens (acumulado: 500)


  [votacoes] página   6 → 100 itens (acumulado: 600)


... (12 outputs left out)

  [votacoes] página  19 → 100 itens (acumulado: 1900)


  [votacoes] página  20 → 100 itens (acumulado: 2000)


  [votacoes] página  21 →   4 itens (acumulado: 2004)
✅ votacoes: 2004 registros em 21 página(s) — 21.1s
(2004, 11)


,id,uri,data,dataHoraRegistro,siglaOrgao,uriOrgao,uriEvento,proposicaoObjeto,uriProposicaoObjeto,descricao,aprovacao
0,2532113-40,https://dadosabertos.camara.leg.br/api/v2/votacoes/2532113-40,2026-11-26,2026-11-26T11:53:03,PLEN,https://dadosabertos.camara.leg.br/api/v2/orgaos/180,https://dadosabertos.camara.leg.br/api/v2/eventos/85534,NaN,NaN,Aprovada a Redação Final assinada pelo Dep. Afonso Florence (PT-BA).,1.0
1,2532113-37,https://dadosabertos.camara.leg.br/api/v2/votacoes/2532113-37,2026-11-26,2026-11-26T11:51:47,PLEN,https://dadosabertos.camara.leg.br/api/v2/orgaos/180,https://dadosabertos.camara.leg.br/api/v2/eventos/85534,NaN,NaN,"Aprovada a Subemenda Substitutiva ao Projeto de Decreto Legislativo nº 366, ...",1.0
2,2516578-35,https://dadosabertos.camara.leg.br/api/v2/votacoes/2516578-35,2026-11-26,2026-11-26T11:44:52,PLEN,https://dadosabertos.camara.leg.br/api/v2/orgaos/180,https://dadosabertos.camara.leg.br/api/v2/eventos/85534,NaN,NaN,Aprovada a Redação Final assinada pelo Dep. Ricardo Ayres (Republicanos/TO).,1.0
3,2516578-33,https://dadosabertos.camara.leg.br/api/v2/votacoes/2516578-33,2026-11-26,2026-11-26T11:44:26,PLEN,https://dadosabertos.camara.leg.br/api/v2/orgaos/180,https://dadosabertos.camara.leg.br/api/v2/eventos/85534,NaN,NaN,"Aprovado o Projeto de Decreto Legislativo nº 235, de 2025.",1.0
4,949156-136,https://dadosabertos.camara.leg.br/api/v2/votacoes/949156-136,2026-11-25,2026-11-25T18:29:04,PLEN,https://dadosabertos.camara.leg.br/api/v2/orgaos/180,https://dadosabertos.camara.leg.br/api/v2/eventos/85492,NaN,NaN,"Aprovada a Redação Final assinada pelo relator, Dep. Thiago de Joaldo (REPUB...",1.0


## 8. Despesas — `/deputados/{id}/despesas`
Despesas são **por deputado**: um `while` de paginação **dentro** de um `for` pelos deputados (~513 × várias páginas).
Para testar rápido, `AMOSTRA_DEPUTADOS` limita a quantidade. Use `None` para rodar todos (demora vários minutos).

In [8]:
AMOSTRA_DEPUTADOS = 10      # None = todos os deputados
ANO_DESPESAS = DATA_FIM.year

ids_deputados = df_deputados["id"].tolist()[:AMOSTRA_DEPUTADOS]
raw_despesas: list[dict] = []

for i, id_dep in enumerate(ids_deputados, start=1):
    print(f"[{i}/{len(ids_deputados)}] deputado {id_dep}")
    despesas_dep = extrair_paginado(
        f"deputados/{id_dep}/despesas",
        params={"ano": ANO_DESPESAS, "ordem": "ASC", "ordenarPor": "ano"},
        verbose=False,
    )
    for d in despesas_dep:
        d["idDeputado"] = id_dep   # a API não devolve o id do deputado no registro
    raw_despesas.extend(despesas_dep)

df_despesas = pd.DataFrame(raw_despesas)
print(df_despesas.shape)
df_despesas.head()

[1/10] deputado 204379


✅ deputados/204379/despesas: 233 registros em 3 página(s) — 0.6s
[2/10] deputado 220714


✅ deputados/220714/despesas: 178 registros em 2 página(s) — 0.3s
[3/10] deputado 221328


✅ deputados/221328/despesas: 376 registros em 4 página(s) — 0.9s
[4/10] deputado 204560


✅ deputados/204560/despesas: 216 registros em 3 página(s) — 0.7s
[5/10] deputado 204528


✅ deputados/204528/despesas: 288 registros em 3 página(s) — 0.6s
[6/10] deputado 121948


... (2 outputs left out)

✅ deputados/160508/despesas: 354 registros em 4 página(s) — 1.0s
[9/10] deputado 136811


✅ deputados/136811/despesas: 811 registros em 9 página(s) — 2.3s
[10/10] deputado 178835


✅ deputados/178835/despesas: 623 registros em 7 página(s) — 1.8s
(3627, 18)


,ano,mes,tipoDespesa,codDocumento,tipoDocumento,codTipoDocumento,dataDocumento,numDocumento,valorDocumento,urlDocumento,nomeFornecedor,cnpjCpfFornecedor,valorLiquido,valorGlosa,numRessarcimento,codLote,parcela,idDeputado
0,2026,1,DIVULGAÇÃO DA ATIVIDADE PARLAMENTAR.,8046356,Nota Fiscal,0,2026-01-29T00:00:00,298,5000.00,https://www.camara.leg.br/cota-parlamentar/documentos/publ/3308/2026/8046356...,VN & FW LTDA-M,22209187000170,-5000.00,0.0,,2206629.0,0.0,204379
1,2026,1,DIVULGAÇÃO DA ATIVIDADE PARLAMENTAR.,8046357,Nota Fiscal,0,2026-01-27T00:00:00,90,6200.00,https://www.camara.leg.br/cota-parlamentar/documentos/publ/3308/2026/8046357...,QUINTELA & SA SERVIÇOS LTDA,31277848000196,-6200.00,0.0,,2206629.0,0.0,204379
2,2026,1,DIVULGAÇÃO DA ATIVIDADE PARLAMENTAR.,8046358,Nota Fiscal,0,2026-02-02T00:00:00,61,1500.00,https://www.camara.leg.br/cota-parlamentar/documentos/publ/3308/2026/8046358...,Eka Comunicacao,58744132000108,-1500.00,0.0,,2206629.0,0.0,204379
3,2026,1,COMBUSTÍVEIS E LUBRIFICANTES.,8054458,DANFE/DACTE,4,2026-01-29T00:00:00,12158,341.99,http://www.camara.leg.br/cota-parlamentar/nota-fiscal-eletronica?ideDocument...,POSTO IPANEMA TEREZOPOLIS I LTDA,62411109000179,-341.99,0.0,,2211266.0,0.0,204379
4,2026,1,COMBUSTÍVEIS E LUBRIFICANTES.,8054468,DANFE/DACTE,4,2026-01-15T00:00:00,4953,303.65,http://www.camara.leg.br/cota-parlamentar/nota-fiscal-eletronica?ideDocument...,POSTO IPANEMA TEREZOPOLIS I LTDA,62411109000179,-303.65,0.0,,2211266.0,0.0,204379


## 9. Resumo da extração

In [9]:
resumo = pd.DataFrame(
    [
        ("partidos", len(df_partidos)),
        ("deputados", len(df_deputados)),
        ("proposicoes", len(df_proposicoes)),
        ("votacoes", len(df_votacoes)),
        (f"despesas ({len(ids_deputados)} deputados)", len(df_despesas)),
    ],
    columns=["endpoint", "registros"],
)
resumo

,endpoint,registros
0,partidos,21
1,deputados,513
2,proposicoes,7514
3,votacoes,2004
4,despesas (10 deputados),3627


## 10. Salvar a camada Bronze (`data/raw/`)
Os retornos são gravados **exatamente como vieram da API** em JSON. O notebook `02_padronizacao.ipynb` lê esses arquivos, então não é preciso extrair de novo para padronizar.
`data/raw/` está no `.gitignore`: os dados brutos não sobem para o GitHub.

In [10]:
import json
from pathlib import Path

PASTA_RAW = Path("../data/raw")
PASTA_RAW.mkdir(parents=True, exist_ok=True)

brutos = {
    "partidos": raw_partidos,
    "deputados": raw_deputados,
    "proposicoes": raw_proposicoes,
    "votacoes": raw_votacoes,
    "despesas": raw_despesas,
}
for nome, registros in brutos.items():
    caminho = PASTA_RAW / f"{nome}.json"
    caminho.write_text(json.dumps(registros, ensure_ascii=False), encoding="utf-8")
    print(f"💾 {caminho} — {len(registros)} registros")

💾 ..\data\raw\partidos.json — 21 registros
💾 ..\data\raw\deputados.json — 513 registros
💾 ..\data\raw\proposicoes.json — 7514 registros
💾 ..\data\raw\votacoes.json — 2004 registros
💾 ..\data\raw\despesas.json — 3627 registros


> **Próximo passo:** `02_padronizacao.ipynb` (snake_case, tipos, datas, deduplicação).